# 🎯 AI Influencer Outreach & Negotiation Agent

Yeh notebook aapke Brand Brief ke basis par ek intelligent **Human-like Creator Talent Manager Agent** banata hai jo WhatsApp/Call ke style me baat karta hai aur rates ko item-by-item confirm karta hai.

### Flow:
1. **Brand Brief Parser**: Brand ki requirements (Instagram Reels, Stories, YouTube videos, budget, timeline) extract karta hai.
2. **Interactive Creator Chat**: Human style me natural, short 1-2 line messages me creator se deal karta hai.
3. **Item-by-Item Price Confirmation**: Har ek deliverable ka rate confirm karke total lock karta hai.
4. **Deal Extractor**: Chat khatam hone par final price aur agreed deliverables ka structured summary/table deta hai.

In [ ]:
# Step 1: Install required libraries
!pip install -q -U google-genai pydantic pandas

In [ ]:
# Step 2: Gemini API Key Setup & Client Initialization
import os
import time
import json
import pandas as pd
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from typing import List, Optional, Dict, Any

try:
    from google.colab import userdata
    api_key = userdata.get('GEMINI_API_KEY')
except Exception:
    import getpass
    api_key = os.getenv('GEMINI_API_KEY') or getpass.getpass('Apna Gemini API Key enter karein: ')

client = genai.Client(api_key=api_key)
MODELS_LIST = ['gemini-3.5-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'gemini-flash-lite-latest']
print('✅ Gemini Client ready with resilient fallback models:', MODELS_LIST)

In [ ]:
# Step 3: Define Schemas for Structured Output

class CampaignBrief(BaseModel):
    brand_name: str = Field(description='Brand ka naam')
    campaign_name: str = Field(description='Campaign theme/title')
    target_audience: str = Field(description='Target audience')
    deliverables_list: List[str] = Field(description='List of deliverables, e.g. 1 Instagram Reel, 2 Stories, 1 YouTube Short')
    timeline: str = Field(description='Live date deadline')
    budget_guideline: Optional[str] = Field(default='Open for quotes', description='Brand budget range')
    key_messaging: str = Field(description='Key USPs aur features')

class DealSummary(BaseModel):
    creator_name: str
    deliverables_agreed: List[str]
    initial_quoted_price: str
    final_agreed_price: str
    status: str # 'Deal Agreed', 'Negotiating', 'Out of Budget', 'Declined'
    negotiation_notes: str
    next_steps: str

In [ ]:
# Step 4: Sample Brand Brief (Aap yahan apna real brief bhi paste kar sakte hain)

raw_brief_text = """
Brand: Bolt Audio
Campaign: Launch of Bolt Mirage Smartwatch (AMOLED Display, BT Calling, 7-day battery)
Target Audience: 18-35 age group, tech enthusiasts & college youth.
Deliverables Required:
- 1 Instagram Reel (30-60 secs) showing unboxing + 3 key features in daily life.
- 2 Instagram Stories (1 announcement sticker, 1 direct buy link + 10% promo code).
- 1 YouTube Shorts showcasing quick first impressions.
Timeline: Must go live within next 10 days.
Budget Guidelines: INR 25,000 to INR 35,000 per creator.
Key Messaging: Super sleek AMOLED screen, seamless Bluetooth calling, durable battery.
"""

print('⏳ Parsing Brand Brief...')
parse_prompt = f"""
Analyze the following brand brief text and extract all campaign details:
{raw_brief_text}
"""

# Safe fallback generate
parsed_brief = None
for m in MODELS_LIST:
    try:
        response = client.models.generate_content(
            model=m,
            contents=parse_prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=CampaignBrief,
                temperature=0.2
            )
        )
        parsed_brief = CampaignBrief.model_validate_json(response.text)
        break
    except Exception:
        continue

print('✅ Brief Parsed Successfully!')
print(f'Brand: {parsed_brief.brand_name}')
print('Deliverables needed:')
for d in parsed_brief.deliverables_list:
    print(f'  • {d}')
print(f'Budget Guideline: {parsed_brief.budget_guideline}')

In [ ]:
# Step 5: Start Interactive Chat Session (Human-like WhatsApp/Call Style)

creator_name = 'narayan'  # Test creator name
creator_niche = 'Tech & Gadgets Reviews'
deliverables_str = ', '.join(parsed_brief.deliverables_list)

system_instruction = f"""
You are 'Aman', Creator Partnerships Manager representing {parsed_brief.brand_name}.
You are chatting with creator '{creator_name}' ({creator_niche}) over WhatsApp / Phone Call.

CRITICAL RULES TO SOUND EXACTLY LIKE A REAL HUMAN:
1. SUPER CONCISE (1 to 2 short sentences max, 15-30 words). Never write long paragraphs or essays.
2. NO BULLET POINTS: Absolutely NEVER use bullet points. Speak casually: 'We need 1 Reel, 2 Stories, and 1 Short for the launch.'
3. NATURAL PACING (Do not dump all info at once):
   - First message: Just a warm, friendly greeting. Compliment their work and ask if they are open for a collab.
   - When creator replies: Casual brief about deliverables and ask for their commercial quote/rate card.
4. MANDATORY ITEM-BY-ITEM PRICE CONFIRMATION:
   - Jab bhi creator price/quote bataye, turant deal accept mat karo.
   - HAMESHA har ek deliverable ka price item-by-item confirm karo ek short sentence me: 
     E.g.: 'Got it! So 1 Reel ka ₹5k, Short ka ₹5k, aur Stories ka ₹3k each — Total ₹16k ho gaya, right? Bas ek baar confirm kar do.'
   - Jab creator confirm kare, tabhi agle steps (delivery address/timeline) par jao.
5. ASK ONLY ONE QUESTION AT A TIME.
6. Tone: Natural Hinglish/English, casual yet professional Indian talent manager.

Internal Campaign Info:
- Brand: {parsed_brief.brand_name}
- Deliverables: {deliverables_str}
- Budget: {parsed_brief.budget_guideline}
- Timeline: {parsed_brief.timeline}
"""

chat_history_llm = []
chat_transcript = []

def send_agent_message(user_msg):
    chat_history_llm.append({"role": "user", "parts": [{"text": user_msg}]})
    for m in MODELS_LIST:
        try:
            res = client.models.generate_content(
                model=m,
                contents=chat_history_llm,
                config=types.GenerateContentConfig(
                    system_instruction=system_instruction,
                    temperature=0.7
                )
            )
            reply = res.text.strip()
            chat_history_llm.append({"role": "model", "parts": [{"text": reply}]})
            return reply
        except Exception:
            time.sleep(1)
            continue
    return 'Sorry, please repeat that.'

# Bot sends first outreach message (natural short opening)
bot_greeting = send_agent_message(f'Send a natural, short 1-2 sentence opening message to {creator_name} ({creator_niche}) introducing yourself from {parsed_brief.brand_name} and asking if they are open for a quick collab.')
print(f'🤖 Bot (Aman):\n{bot_greeting}\n')
chat_transcript.append({'role': 'agent', 'text': bot_greeting})

print("💡 TIP: Type 'exit' to end the chat and see the final deal summary.\n")

while True:
    user_reply = input(f'👤 You ({creator_name}): ').strip()
    if not user_reply:
        continue
    if user_reply.lower() in ['exit', 'quit']:
        print('\nChat concluded.')
        break
    
    chat_transcript.append({'role': 'creator', 'text': user_reply})
    bot_reply = send_agent_message(user_reply)
    print(f'\n🤖 Bot (Aman):\n{bot_reply}\n')
    chat_transcript.append({'role': 'agent', 'text': bot_reply})

In [ ]:
# Step 6: Extract Final Deal Summary & Export to Table

transcript_text = '\n'.join([f"{m['role'].upper()}: {m['text']}" for m in chat_transcript])

extract_prompt = f"""
Analyze the following negotiation transcript between Brand Manager and Creator '{creator_name}' for {parsed_brief.brand_name}.
Extract deal status, deliverables, initial quote, and final agreed price.

Transcript:
{transcript_text}
"""

deal = None
for m in MODELS_LIST:
    try:
        summary_response = client.models.generate_content(
            model=m,
            contents=extract_prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=DealSummary,
                temperature=0.1
            )
        )
        deal = DealSummary.model_validate_json(summary_response.text)
        break
    except Exception:
        continue

if deal:
    df = pd.DataFrame([{
        'Creator': deal.creator_name,
        'Status': deal.status,
        'Initial Quote': deal.initial_quoted_price,
        'Final Agreed Price': deal.final_agreed_price,
        'Deliverables': ', '.join(deal.deliverables_agreed),
        'Notes': deal.negotiation_notes,
        'Next Steps': deal.next_steps
}])
    print('📊 FINAL DEAL RECORD:')
    display(df)